# Proyecto 1 — Monitoreo transaccional: detectar lo que el orden revela

**Deep Learning — Universidad del Valle, 2026**

Adrián González (23152) · Jose Ordoñez (231329)

El comité de riesgos de Banco del Altiplano nos pregunta si el orden de las transacciones aporta información que las variables agregadas no capturan, bajo qué condiciones y cuánto vale esa información en quetzales.

En este notebook construimos datos sintéticos con cinco mecanismos de fraude, comparamos un modelo sin orden (A) contra uno secuencial (B, GRU), hacemos dos pruebas para aislar el valor del orden, proponemos un híbrido (C) y llegamos a una decisión económica con umbral óptimo.

In [18]:
import json
import os
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from sklearn.metrics import (average_precision_score, precision_recall_curve,
                             precision_score, recall_score, f1_score)
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
import pickle

# ---------------- Parametros globales del experimento ----------------
SEED = 42
W_COMPLETO = 10
W_RECORTADO = 2
COSTO_FN = 4200.0   # Q por fraude no detectado
COSTO_FP = 180.0    # Q por bloquear transaccion legitima

# Rutas de salida (relativas al directorio donde vive el notebook)
BASE = os.path.dirname(os.path.abspath("__file__")) if "__file__" in dir() else "."
RES_DIR = os.path.join(BASE, "resultados")
FIG_DIR = os.path.join(BASE, "figuras")
ART_DIR = os.path.join(BASE, "artefactos")
for d in (RES_DIR, FIG_DIR, ART_DIR):
    os.makedirs(d, exist_ok=True)

np.random.seed(SEED)
plt.rcParams['figure.dpi'] = 100
print('Semilla del experimento:', SEED)


Semilla del experimento: 42


## 1. Datos: Ruta A — generador propio con semilla fija

Simulamos tarjetas de un banco (a escala reducida, pero preservando la mecánica del problema real). Cada cliente tiene su propio perfil de gasto: monto típico, comercios preferidos, canal habitual, horario. Sobre ese fondo normal inyectamos cinco mecanismos de fraude, elegidos para que no todos dependan del orden en la misma medida.

`escalada_rapida` usa varias compras pequeñas de prueba seguidas de una compra grande en minutos — depende del orden, al menos en teoría. `escalada_lenta` hace lo mismo pero repartido en varios días, también depende del orden y requiere más historia. `cambio_canal` es donde el orden importa de verdad: el cliente casi siempre usa un canal, y el fraude oscila POS → online → ATM → online en minutos. `monto_atipico` es un caso control: una sola transacción con monto muy superior al habitual, sin nada raro alrededor, no debería depender del orden. Y `fraude_sigiloso` es el caso difícil: el monto es normal, solo cambia el comercio y el canal menos usados, y es el mecanismo donde esperamos que el modelo falle más.

El generador es determinístico: la misma semilla produce siempre el mismo dataset.

In [19]:
MERCHANT_CATS = ["comida", "retail", "viajes", "entretenimiento", "servicios"]
CHANNELS = ["POS", "online", "ATM"]


def generar_datos(seed=SEED, n_clientes=3000, dias=45, frac_fraude=0.10):
    """Generador sintetico reproducible: misma semilla -> mismos datos."""
    rng = np.random.default_rng(seed)
    registros = []

    # perfiles por cliente
    lam = rng.gamma(shape=2.0, scale=0.15, size=n_clientes)
    mu_amt = rng.normal(4.5, 0.5, size=n_clientes)
    sigma_amt = rng.uniform(0.3, 0.6, size=n_clientes)
    merch_pref = rng.dirichlet(np.ones(len(MERCHANT_CATS)) * 2, size=n_clientes)
    canal_pref = rng.dirichlet([8, 2.5, 0.8], size=n_clientes)
    hora_media = rng.normal(15, 3, size=n_clientes) % 24

    # seleccionar clientes con fraude
    n_fraude = int(n_clientes * frac_fraude)
    clientes_fraude = rng.choice(n_clientes, size=n_fraude, replace=False)
    tipos = rng.choice(
        ["escalada_rapida", "escalada_lenta", "cambio_canal", "monto_atipico", "fraude_sigiloso"],
        size=n_fraude, p=[0.35, 0.15, 0.20, 0.15, 0.15])
    tipo_por_cliente = dict(zip(clientes_fraude, tipos))
    dia_fraude = dict(zip(clientes_fraude, rng.uniform(6, dias - 6, size=n_fraude)))

    tx_id = 0
    for c in range(n_clientes):
        n_tx_esperadas = max(3, int(rng.poisson(lam[c] * dias)))
        dias_tx = np.sort(rng.uniform(0, dias, size=n_tx_esperadas))
        merchant_hist = set()

        for d in dias_tx:
            hora = (rng.normal(hora_media[c], 2.5)) % 24
            cat = rng.choice(MERCHANT_CATS, p=merch_pref[c])
            canal = rng.choice(CHANNELS, p=canal_pref[c])
            monto = float(np.exp(rng.normal(mu_amt[c], sigma_amt[c])))
            nuevo = cat + canal not in merchant_hist
            merchant_hist.add(cat + canal)
            registros.append([tx_id, c, d, hora, monto, canal, cat, int(nuevo), 0, "legitima"])
            tx_id += 1

        # inyectar episodio de fraude
        if c in tipo_por_cliente:
            tipo = tipo_por_cliente[c]
            t0 = dia_fraude[c]
            base_amt = float(np.exp(mu_amt[c]))

            if tipo == "escalada_rapida":
                n_test = rng.integers(3, 6)
                for k in range(n_test):
                    d = t0 + k * (rng.uniform(1, 6) / (24 * 60))
                    monto = float(rng.uniform(5, 50))
                    registros.append([tx_id, c, d, (rng.uniform(0, 24)), monto, "online",
                                      rng.choice(MERCHANT_CATS), 1, 0, "escalada_rapida"])
                    tx_id += 1
                d_final = t0 + n_test * (rng.uniform(1, 6) / (24 * 60)) + rng.uniform(2, 8) / (24 * 60)
                monto_final = base_amt * rng.uniform(8, 20)
                registros.append([tx_id, c, d_final, (rng.uniform(0, 24)), monto_final, "online",
                                  rng.choice(MERCHANT_CATS), 1, 1, "escalada_rapida"])
                tx_id += 1

            elif tipo == "escalada_lenta":
                n_test = rng.integers(3, 6)
                for k in range(n_test):
                    d = t0 + k * rng.uniform(0.5, 1.5)
                    monto = float(np.exp(rng.normal(mu_amt[c], sigma_amt[c])) * rng.uniform(0.3, 0.8))
                    registros.append([tx_id, c, d, (rng.normal(hora_media[c], 2.5)) % 24, monto,
                                      rng.choice(CHANNELS, p=canal_pref[c]),
                                      rng.choice(MERCHANT_CATS, p=merch_pref[c]), 0, 0, "escalada_lenta"])
                    tx_id += 1
                d_final = t0 + n_test * rng.uniform(0.5, 1.5) + rng.uniform(0.5, 1.5)
                monto_final = base_amt * rng.uniform(8, 20)
                registros.append([tx_id, c, d_final, (rng.uniform(0, 24)), monto_final, "online",
                                  rng.choice(MERCHANT_CATS), 1, 1, "escalada_lenta"])
                tx_id += 1

            elif tipo == "cambio_canal":
                secuencia_canal = ["POS", "online", "ATM", "online"]
                for k, ch in enumerate(secuencia_canal):
                    d = t0 + k * (rng.uniform(3, 15) / (24 * 60))
                    monto = base_amt * rng.uniform(0.8, 2.5)
                    es_ultimo = int(k == len(secuencia_canal) - 1)
                    registros.append([tx_id, c, d, (rng.uniform(0, 24)), monto, ch,
                                      rng.choice(MERCHANT_CATS), 0, es_ultimo, "cambio_canal"])
                    tx_id += 1

            elif tipo == "monto_atipico":
                monto = base_amt * rng.uniform(15, 30)
                registros.append([tx_id, c, t0, (rng.normal(hora_media[c], 2.5)) % 24, monto,
                                  rng.choice(CHANNELS, p=canal_pref[c]),
                                  rng.choice(MERCHANT_CATS, p=merch_pref[c]), 1, 1, "monto_atipico"])
                tx_id += 1

            elif tipo == "fraude_sigiloso":
                monto = base_amt * rng.uniform(0.7, 1.4)
                canal_raro = CHANNELS[int(np.argmin(canal_pref[c]))]
                comercio_raro = MERCHANT_CATS[int(np.argmin(merch_pref[c]))]
                hora_rara = (hora_media[c] + rng.uniform(9, 15)) % 24
                registros.append([tx_id, c, t0, hora_rara, monto, canal_raro, comercio_raro,
                                  1, 1, "fraude_sigiloso"])
                tx_id += 1

    df = pd.DataFrame(registros, columns=[
        "tx_id", "cliente_id", "dia", "hora", "monto", "canal", "categoria",
        "es_comercio_nuevo", "label_fraude", "tipo_fraude"])
    df = df.sort_values(["dia"]).reset_index(drop=True)
    df["tx_id"] = np.arange(len(df))
    return df


df = generar_datos(seed=SEED)
print('Transacciones totales:', len(df))
print('Tasa de fraude global: {:.3%}'.format(df['label_fraude'].mean()))
resumen = df.groupby('tipo_fraude').agg(n_transacciones=('tx_id','size'),
                                          n_positivas=('label_fraude','sum')).reset_index()
resumen


Transacciones totales: 41975
Tasa de fraude global: 0.715%


,tipo_fraude,n_transacciones,n_positivas
0,cambio_canal,220,55
1,escalada_lenta,218,43
2,escalada_rapida,518,104
3,fraude_sigiloso,50,50
4,legitima,40921,0
5,monto_atipico,48,48


### Partición temporal

La separación respeta el tiempo: 60% de los días más antiguos para entrenar, el siguiente 20% para validar y el último 20% para prueba. El set de prueba se toca una sola vez, ya con el modelo y el umbral fijos.

In [20]:
# Features por cliente: promedio de monto 24h, maximo del dia, tx en 1h/24h,
# diversidad de comercios y canales en 7 dias, ratio contra historico, etc.
# Todo se calcula usando SOLO el pasado de cada cliente (sin fuga de informacion).

def _features_cliente(g):
    """g: dataframe de un cliente, ordenado por 'dia'. Variables agregadas causales."""
    dias = g["dia"].values
    montos = g["monto"].values
    canales = g["canal"].values
    cats = g["categoria"].values
    n = len(g)

    monto_prom_24h = np.zeros(n); monto_max_dia = np.zeros(n)
    n_tx_1h = np.zeros(n); n_tx_24h = np.zeros(n)
    div_comercio_7d = np.zeros(n); div_canal_7d = np.zeros(n)
    t_desde_ultima = np.zeros(n); ratio_hist = np.zeros(n); n_hist = np.zeros(n)
    hist_sum = 0.0; hist_n = 0

    for i in range(n):
        t = dias[i]
        w24 = (dias <= t) & (dias > t - 1.0)
        w7 = (dias <= t) & (dias > t - 7.0)
        w1h = (dias <= t) & (dias > t - (1.0 / 24.0))
        wday = (np.floor(dias) == np.floor(t)) & (dias <= t)

        monto_prom_24h[i] = montos[w24].mean()
        monto_max_dia[i] = montos[wday].max() if wday.any() else 0.0
        n_tx_1h[i] = w1h.sum(); n_tx_24h[i] = w24.sum()
        div_comercio_7d[i] = len(set(cats[w7])); div_canal_7d[i] = len(set(canales[w7]))
        t_desde_ultima[i] = (t - dias[i - 1]) * 24 if i > 0 else 24.0 * 7
        ratio_hist[i] = montos[i] / (hist_sum / hist_n) if hist_n > 0 else 1.0
        n_hist[i] = hist_n
        hist_sum += montos[i]; hist_n += 1

    out = g.copy()
    out["monto_prom_24h"] = monto_prom_24h; out["monto_max_dia"] = monto_max_dia
    out["n_tx_1h"] = n_tx_1h; out["n_tx_24h"] = n_tx_24h
    out["div_comercio_7d"] = div_comercio_7d; out["div_canal_7d"] = div_canal_7d
    out["horas_desde_ultima"] = t_desde_ultima
    out["ratio_monto_hist"] = ratio_hist; out["n_tx_historicas"] = n_hist
    return out


def agregar_variables_causales(df):
    df = df.sort_values(["cliente_id", "dia"]).reset_index(drop=True)
    partes = [_features_cliente(g) for _, g in df.groupby("cliente_id", sort=False)]
    out = pd.concat(partes, ignore_index=True)
    out["dow"] = np.floor(out["dia"]).astype(int) % 7
    out["es_fin_semana"] = (out["dow"] >= 5).astype(int)
    return out.sort_values("tx_id").reset_index(drop=True)


AGG_COLS = [
    "monto", "monto_prom_24h", "monto_max_dia", "n_tx_1h", "n_tx_24h",
    "div_comercio_7d", "div_canal_7d", "horas_desde_ultima", "ratio_monto_hist",
    "n_tx_historicas", "hora", "es_fin_semana", "es_comercio_nuevo",
]


def matriz_agregada(df):
    X = df[AGG_COLS].copy()
    X["log_monto"] = np.log1p(X["monto"]); X.drop(columns=["monto"], inplace=True)
    X["log_monto_prom_24h"] = np.log1p(X["monto_prom_24h"]); X.drop(columns=["monto_prom_24h"], inplace=True)
    X["log_monto_max_dia"] = np.log1p(X["monto_max_dia"]); X.drop(columns=["monto_max_dia"], inplace=True)
    X["log_horas_desde_ultima"] = np.log1p(X["horas_desde_ultima"]); X.drop(columns=["horas_desde_ultima"], inplace=True)
    return X


def _vector_paso(monto, hora, canal, categoria, es_nuevo, horas_desde_ultima):
    v = np.zeros(14, dtype=np.float32)
    v[0] = np.log1p(monto)
    v[1] = np.sin(2 * np.pi * hora / 24); v[2] = np.cos(2 * np.pi * hora / 24)
    v[3] = np.log1p(horas_desde_ultima); v[4] = float(es_nuevo)
    v[5 + CHANNELS.index(canal)] = 1.0
    v[8 + MERCHANT_CATS.index(categoria)] = 1.0
    return v


def construir_secuencias(df, W=W_COMPLETO):
    """(N, W, 14) con relleno a la izquierda (sin filtracion de futuro)."""
    df = df.sort_values(["cliente_id", "dia"]).reset_index(drop=True)
    N = len(df); X_seq = np.zeros((N, W, 14), dtype=np.float32)
    for _, g in df.groupby("cliente_id", sort=False):
        g = g.reset_index()
        pasos = []
        for _, row in g.iterrows():
            v = _vector_paso(row["monto"], row["hora"], row["canal"], row["categoria"],
                             row["es_comercio_nuevo"], row["horas_desde_ultima"])
            pasos.append(v)
            hist = pasos[-W:]
            seq = np.zeros((W, 14), dtype=np.float32)
            seq[W - len(hist):] = np.array(hist)
            X_seq[row["index"]] = seq
    orden = df["tx_id"].values
    X_final = np.zeros_like(X_seq)
    X_final[orden] = X_seq
    return X_final


# ---------------- Construccion y particion temporal ----------------
dfa = agregar_variables_causales(df).sort_values('tx_id').reset_index(drop=True)
Xagg_df = matriz_agregada(dfa)
Xseq_full = construir_secuencias(dfa, W=W_COMPLETO)
Xseq_short = construir_secuencias(dfa, W=W_RECORTADO)

dias = dfa['dia'].values
d_train_end = np.quantile(dias, 0.60)
d_val_end = np.quantile(dias, 0.80)
split = np.where(dias <= d_train_end, 'train', np.where(dias <= d_val_end, 'val', 'test'))
y = dfa['label_fraude'].values.astype(float)
tipo = dfa['tipo_fraude'].values

print(f"train: {(split=='train').sum()}  val: {(split=='val').sum()}  test: {(split=='test').sum()}")
print(f"corte train/val en el dia {d_train_end:.1f}, corte val/test en el dia {d_val_end:.1f}")
tr, va, te = split=='train', split=='val', split=='test'


train: 25185  val: 8395  test: 8395
corte train/val en el dia 26.9, corte val/test en el dia 35.8


Las variables agregadas se escalan con un `StandardScaler` ajustado solo con el set de entrenamiento. Lo mismo para las variables continuas de las secuencias (monto y tiempo desde la última transacción). Ningún estadístico de validación o prueba se usa para normalizar.

In [21]:
def escalar_agg(Xagg_df, split):
    scaler = StandardScaler()
    scaler.fit(Xagg_df[split == "train"].values)
    return scaler.transform(Xagg_df.values), scaler


def escalar_seq(Xseq, split):
    """Escala solo las columnas continuas (0: log_monto, 3: log_horas_desde_ultima)
    usando estadisticas SOLO de pasos validos del set de train."""
    Xs = Xseq.copy()
    tr_idx = np.where(split == "train")[0]
    valido = np.abs(Xseq[tr_idx]).sum(axis=2) > 0
    for col in [0, 3]:
        vals = Xseq[tr_idx, :, col]
        mu = vals[valido].mean(); sd = vals[valido].std() + 1e-6
        Xs[:, :, col] = (Xseq[:, :, col] - mu) / sd
        pad = np.abs(Xseq).sum(axis=2) == 0
        Xs[:, :, col][pad] = 0.0
    return Xs


def metrics_en(y_true, p, umbral=0.5):
    pred = (p >= umbral).astype(int)
    return dict(
        auc_pr=float(average_precision_score(y_true, p)),
        precision=float(precision_score(y_true, pred, zero_division=0)),
        recall=float(recall_score(y_true, pred, zero_division=0)),
        f1=float(f1_score(y_true, pred, zero_division=0)),
        n_pos=int(y_true.sum()), n=int(len(y_true)))


Xagg, scaler_agg = escalar_agg(Xagg_df, split)
Xseq_full_e = escalar_seq(Xseq_full, split)
Xseq_short_e = escalar_seq(Xseq_short, split)
print('Variables agregadas (Modelo A):', list(Xagg_df.columns))
print('Forma de las secuencias (Modelo B), W=10:', Xseq_full_e.shape)


Variables agregadas (Modelo A): ['n_tx_1h', 'n_tx_24h', 'div_comercio_7d', 'div_canal_7d', 'ratio_monto_hist', 'n_tx_historicas', 'hora', 'es_fin_semana', 'es_comercio_nuevo', 'log_monto', 'log_monto_prom_24h', 'log_monto_max_dia', 'log_horas_desde_ultima']
Forma de las secuencias (Modelo B), W=10: (41975, 10, 14)


## 2. Núcleo común

### Modelo A — línea base sin orden

Un `HistGradientBoostingClassifier` de scikit-learn sobre las variables agregadas: monto promedio 24h, monto máximo del día, conteo de transacciones por hora, diversidad de comercios/canales en los últimos 7 días, razón contra el promedio histórico del cliente, etc. Representa el techo de lo alcanzable sin leer la secuencia.

In [22]:
def entrenar_modelo_A(Xagg, y, split):
    tr, va = split == "train", split == "val"
    modelo = HistGradientBoostingClassifier(
        max_iter=300, learning_rate=0.06, max_depth=4, l2_regularization=0.1,
        class_weight="balanced", random_state=SEED, validation_fraction=None)
    modelo.fit(Xagg[tr], y[tr])
    p_val = modelo.predict_proba(Xagg[va])[:, 1]
    return modelo, p_val


modelo_A, pA_val = entrenar_modelo_A(Xagg, y, split)
pA_test = modelo_A.predict_proba(Xagg[te])[:, 1]
print('Modelo A -- AUC-PR validacion: {:.4f}'.format(average_precision_score(y[va], pA_val)))
metrics_en(y[va], pA_val)


Modelo A -- AUC-PR validacion: 0.8691


{'auc_pr': 0.8690827056394511,
 'precision': 0.6336633663366337,
 'recall': 0.8533333333333334,
 'f1': 0.7272727272727273,
 'n_pos': 75,
 'n': 8395}

### Modelo B — secuencial (GRU)

Un GRU binario implementado con NumPy (forward + backpropagation through time + Adam), verificado con gradiente numérico antes de entrenar. El modelo es pequeño (24 unidades ocultas) y entrena en segundos sobre CPU.

Recibe la secuencia ordenada de hasta 10 transacciones previas de cada cliente (monto, hora cíclica, canal, comercio, tiempo desde la anterior, si el comercio es nuevo), rellenada a la izquierda cuando el cliente tiene menos historial.

In [23]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -30, 30)))


class GRUBinario:
    """GRU binario en NumPy (forward + BPTT + Adam)."""
    def __init__(self, input_dim, hidden_dim=24, seed=0):
        rng = np.random.default_rng(seed)
        s = 1.0 / np.sqrt(hidden_dim)
        self.H = hidden_dim; self.F = input_dim
        self.Wz = rng.uniform(-s, s, (input_dim, hidden_dim)); self.Uz = rng.uniform(-s, s, (hidden_dim, hidden_dim)); self.bz = np.zeros(hidden_dim)
        self.Wr = rng.uniform(-s, s, (input_dim, hidden_dim)); self.Ur = rng.uniform(-s, s, (hidden_dim, hidden_dim)); self.br = np.zeros(hidden_dim)
        self.Wh = rng.uniform(-s, s, (input_dim, hidden_dim)); self.Uh = rng.uniform(-s, s, (hidden_dim, hidden_dim)); self.bh = np.zeros(hidden_dim)
        self.Wo = rng.uniform(-s, s, (hidden_dim, 1)); self.bo = np.zeros(1)
        self.params = ["Wz", "Uz", "bz", "Wr", "Ur", "br", "Wh", "Uh", "bh", "Wo", "bo"]
        self.m = {p: np.zeros_like(getattr(self, p)) for p in self.params}
        self.v = {p: np.zeros_like(getattr(self, p)) for p in self.params}
        self.t = 0

    def forward(self, X):
        B, T, F = X.shape
        h = np.zeros((B, self.H))
        cache = {"x": [], "z": [], "r": [], "hh": [], "h": [h.copy()]}
        for t in range(T):
            xt = X[:, t, :]
            z = sigmoid(xt @ self.Wz + h @ self.Uz + self.bz)
            r = sigmoid(xt @ self.Wr + h @ self.Ur + self.br)
            hh = np.tanh(xt @ self.Wh + (r * h) @ self.Uh + self.bh)
            h = (1 - z) * h + z * hh
            cache["x"].append(xt); cache["z"].append(z); cache["r"].append(r); cache["hh"].append(hh)
            cache["h"].append(h.copy())
        logits = (h @ self.Wo + self.bo).ravel()
        cache["h_final"] = h; cache["logits"] = logits
        return sigmoid(logits), cache

    def hidden_states(self, X, batch=2048):
        out = np.zeros((len(X), self.H))
        for i in range(0, len(X), batch):
            _, cache = self.forward(X[i:i + batch])
            out[i:i + batch] = cache["h_final"]
        return out

    def predict_proba(self, X, batch=2048):
        out = np.zeros(len(X))
        for i in range(0, len(X), batch):
            p, _ = self.forward(X[i:i + batch]); out[i:i + batch] = p
        return out

    def backward(self, cache, y, pos_weight=1.0, l2=1e-4):
        T = len(cache["x"]); B = y.shape[0]
        h_final = cache["h_final"]
        p = sigmoid(cache["logits"])
        sw = np.where(y == 1, pos_weight, 1.0)
        dlogits = (p - y) * sw / B
        grads = {p_: np.zeros_like(getattr(self, p_)) for p_ in self.params}
        grads["Wo"] = h_final.T @ dlogits.reshape(-1, 1)
        grads["bo"] = np.array([dlogits.sum()])
        dh = dlogits.reshape(-1, 1) @ self.Wo.T

        for t in reversed(range(T)):
            x = cache["x"][t]; z = cache["z"][t]; r = cache["r"][t]
            hh = cache["hh"][t]; h_prev = cache["h"][t]
            dz = dh * (hh - h_prev); dhh = dh * z; dh_prev = dh * (1 - z)
            dhh_raw = dhh * (1 - hh ** 2)
            grads["Wh"] += x.T @ dhh_raw; grads["Uh"] += (r * h_prev).T @ dhh_raw; grads["bh"] += dhh_raw.sum(axis=0)
            dr_hprev = dhh_raw @ self.Uh.T; dr = dr_hprev * h_prev; dh_prev += dr_hprev * r
            dz_raw = dz * z * (1 - z)
            grads["Wz"] += x.T @ dz_raw; grads["Uz"] += h_prev.T @ dz_raw; grads["bz"] += dz_raw.sum(axis=0)
            dh_prev += dz_raw @ self.Uz.T
            dr_raw = dr * r * (1 - r)
            grads["Wr"] += x.T @ dr_raw; grads["Ur"] += h_prev.T @ dr_raw; grads["br"] += dr_raw.sum(axis=0)
            dh_prev += dr_raw @ self.Ur.T
            dh = dh_prev

        for p_ in self.params:
            g = getattr(self, p_)
            if g.ndim == 2: grads[p_] += l2 * g
        return grads

    def adam_step(self, grads, lr=0.01, b1=0.9, b2=0.999, eps=1e-8):
        self.t += 1
        for p in self.params:
            g = grads[p]
            self.m[p] = b1 * self.m[p] + (1 - b1) * g
            self.v[p] = b2 * self.v[p] + (1 - b2) * (g ** 2)
            mhat = self.m[p] / (1 - b1 ** self.t)
            vhat = self.v[p] / (1 - b2 ** self.t)
            setattr(self, p, getattr(self, p) - lr * mhat / (np.sqrt(vhat) + eps))

    def bce_loss(self, p, y, pos_weight=1.0):
        eps = 1e-9
        w = np.where(y == 1, pos_weight, 1.0)
        return float(np.mean(-w * (y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps))))

    def fit(self, X, y, X_val=None, y_val=None, epochs=25, batch_size=256, lr=0.01,
            pos_weight=None, seed=0, verbose=True, log=None):
        rng = np.random.default_rng(seed)
        if pos_weight is None: pos_weight = (y == 0).sum() / max((y == 1).sum(), 1)
        n = len(X); historial = []
        for ep in range(epochs):
            idx = rng.permutation(n); perdida_ep = 0.0
            for i in range(0, n, batch_size):
                bidx = idx[i:i + batch_size]
                p, cache = self.forward(X[bidx])
                grads = self.backward(cache, y[bidx], pos_weight=pos_weight)
                self.adam_step(grads, lr=lr)
                perdida_ep += self.bce_loss(p, y[bidx], pos_weight) * len(bidx)
            perdida_ep /= n
            fila = {"epoch": ep, "loss": perdida_ep}
            if X_val is not None:
                fila["val_auc_pr"] = average_precision_score(y_val, self.predict_proba(X_val))
            historial.append(fila)
            if verbose and (ep % 5 == 0 or ep == epochs - 1):
                msg = f"  epoch {ep:2d}  loss={perdida_ep:.4f}" + (f"  val_AUC-PR={fila['val_auc_pr']:.4f}" if X_val is not None else "")
                print(msg)
                if log is not None: log.append(msg)
        return historial


def entrenar_modelo_B(Xseq, y, split, hidden=24, epochs=30, lr=0.02, seed=SEED, log=None):
    tr, va = split == "train", split == "val"
    modelo = GRUBinario(input_dim=Xseq.shape[2], hidden_dim=hidden, seed=seed)
    hist = modelo.fit(Xseq[tr], y[tr], Xseq[va], y[va], epochs=epochs, lr=lr, batch_size=256, log=log)
    p_val = modelo.predict_proba(Xseq[va])
    return modelo, p_val, hist


In [24]:
def _chequeo_gradiente():
    """Verifica el backprop del GRU con gradiente numerico (error ~1e-8)."""
    rng = np.random.default_rng(0)
    B, T, F = 5, 4, 3
    X = rng.normal(size=(B, T, F)); y = rng.integers(0, 2, size=B).astype(float)
    m = GRUBinario(input_dim=F, hidden_dim=6, seed=1)
    p, cache = m.forward(X)
    grads = m.backward(cache, y, pos_weight=1.3, l2=0.0)

    def loss_fn():
        p, _ = m.forward(X); return m.bce_loss(p, y, pos_weight=1.3)

    eps = 1e-5; max_err = 0.0
    for pname in ["Wz", "Uh", "Wo", "bo", "br"]:
        arr = getattr(m, pname); g_analytic = grads[pname]
        for _ in range(4):
            idx = tuple(rng.integers(0, s) for s in arr.shape)
            orig = arr[idx]
            arr[idx] = orig + eps; l1 = loss_fn()
            arr[idx] = orig - eps; l2 = loss_fn()
            arr[idx] = orig
            g_num = (l1 - l2) / (2 * eps)
            err = abs(g_num - g_analytic[idx]) / (abs(g_num) + abs(g_analytic[idx]) + 1e-8)
            max_err = max(max_err, err)
    return max_err


err = _chequeo_gradiente()
print('Error relativo maximo del backprop:', err)
assert err < 1e-4, 'El gradiente analitico no coincide con el numerico'
print('OK: backprop del GRU verificado.')


Error relativo maximo del backprop: 1.740216417557992e-08
OK: backprop del GRU verificado.


In [25]:
modelo_B, pB_val, hist_B = entrenar_modelo_B(Xseq_full_e, y, split, hidden=24, epochs=30, lr=0.02)
pB_test = modelo_B.predict_proba(Xseq_full_e[te])
auc_B = average_precision_score(y[va], pB_val)
print('Modelo B -- AUC-PR validacion: {:.4f}'.format(auc_B))
metrics_en(y[va], pB_val)


  epoch  0  loss=0.4311  val_AUC-PR=0.7169
  epoch  5  loss=0.0963  val_AUC-PR=0.8738
  epoch 10  loss=0.0683  val_AUC-PR=0.8621
  epoch 15  loss=0.0406  val_AUC-PR=0.8974
  epoch 20  loss=0.0106  val_AUC-PR=0.9178
  epoch 25  loss=0.0065  val_AUC-PR=0.9278
  epoch 29  loss=0.0139  val_AUC-PR=0.9019
Modelo B -- AUC-PR validacion: 0.9019


{'auc_pr': 0.9018977296663026,
 'precision': 0.6194690265486725,
 'recall': 0.9333333333333333,
 'f1': 0.7446808510638298,
 'n_pos': 75,
 'n': 8395}

In [26]:
prec_a, rec_a, _ = precision_recall_curve(y[va], pA_val)
prec_b, rec_b, _ = precision_recall_curve(y[va], pB_val)
plt.figure(figsize=(6,4.5))
plt.plot(rec_a, prec_a, label=f'A (agregado)  AP={average_precision_score(y[va],pA_val):.3f}')
plt.plot(rec_b, prec_b, label=f'B (GRU secuencial)  AP={auc_B:.3f}')
plt.xlabel('Exhaustividad'); plt.ylabel('Precision'); plt.title('A vs B en validacion')
plt.legend(); plt.grid(alpha=0.3); plt.show()


/var/folders/jm/6cz0pdzj2dzf6383_3d80ysh0000gn/T/ipykernel_16626/3125604517.py:7: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.legend(); plt.grid(alpha=0.3); plt.show()


El modelo secuencial B supera a A en AUC-PR de validación, lo cual sugiere que el orden ayuda. Pero esa diferencia por sí sola no es conclusiva.

## 3. Dos pruebas de falsificación

### Prueba 1 — Permutación controlada

Barajamos el orden de la historia de cada secuencia (todas las transacciones previas, sin tocar la transacción evaluada ni las variables agregadas) y evaluamos el modelo B ya entrenado sin reentrenarlo. Si el modelo realmente usa el orden, su desempeño debería caer.

In [27]:
def permutar_historia(Xseq, rng_seed=0):
    """Baraja el orden de los pasos de HISTORIA (todos menos el ultimo), dejando fijos
    los eventos y la transaccion evaluada."""
    rng = np.random.default_rng(rng_seed)
    Xp = Xseq.copy(); T = Xseq.shape[1]
    for i in range(len(Xseq)):
        pad_mask = np.abs(Xseq[i, :T - 1]).sum(axis=1) > 0
        idx_validos = np.where(pad_mask)[0]
        if len(idx_validos) > 1:
            Xp[i, idx_validos] = Xseq[i, rng.permutation(idx_validos)]
    return Xp


def por_tipo(y, tipo, p, split_mask):
    """Score medio por mecanismo de fraude. y/tipo globales, p recortado al subconjunto."""
    out = {}
    tipo_sub = tipo[split_mask]; y_sub = y[split_mask]
    for t in [t for t in np.unique(tipo) if t != "legitima"]:
        sel = tipo_sub == t
        if sel.sum() == 0: continue
        y_t = y_sub[sel]; p_t = p[sel]
        if y_t.sum() == 0: continue
        out[t] = {"n": int(y_t.sum()), "score_medio": float(p_t[y_t == 1].mean())}
    return out


# Permutar el orden de la historia, dejar fija la tx evaluada
Xseq_val_perm = permutar_historia(Xseq_full_e[va], rng_seed=1)
pB_val_perm = modelo_B.predict_proba(Xseq_val_perm)
auc_perm = average_precision_score(y[va], pB_val_perm)
print(f'AUC-PR orden original : {auc_B:.4f}')
print(f'AUC-PR historia barajada: {auc_perm:.4f}')
print(f'Caida relativa: {100*(auc_B-auc_perm)/auc_B:.1f}%')

por_tipo_orig = por_tipo(y, tipo, pB_val, va)
por_tipo_perm = por_tipo(y, tipo, pB_val_perm, va)
tabla = pd.DataFrame({
    'score_medio_original': {k: v['score_medio'] for k,v in por_tipo_orig.items()},
    'score_medio_permutado': {k: v['score_medio'] for k,v in por_tipo_perm.items()},
})
tabla['caida'] = tabla['score_medio_original'] - tabla['score_medio_permutado']
tabla.sort_values('caida', ascending=False)


AUC-PR orden original : 0.9019
AUC-PR historia barajada: 0.8193
Caida relativa: 9.2%


,score_medio_original,score_medio_permutado,caida
cambio_canal,0.850320,0.318205,0.532115
monto_atipico,0.905707,0.899498,0.006209
escalada_rapida,0.999993,0.999507,0.000485
escalada_lenta,0.991104,0.997666,-0.006562
fraude_sigiloso,0.788188,0.861485,-0.073298


In [28]:
tipos_frau = list(tabla.index)
x = np.arange(len(tipos_frau)); w=0.35
plt.figure(figsize=(7,4.2))
plt.bar(x-w/2, tabla['score_medio_original'], w, label='Orden original')
plt.bar(x+w/2, tabla['score_medio_permutado'], w, label='Historia permutada')
plt.xticks(x, tipos_frau, rotation=15); plt.ylabel('Score promedio (positivos)')
plt.title('Sensibilidad al orden por mecanismo de fraude'); plt.legend(); plt.show()


/var/folders/jm/6cz0pdzj2dzf6383_3d80ysh0000gn/T/ipykernel_16626/3282980106.py:7: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.title('Sensibilidad al orden por mecanismo de fraude'); plt.legend(); plt.show()


El desplome más fuerte ocurre en `cambio_canal` (el mecanismo diseñado explícitamente como una secuencia de transiciones): su score promedio cae de 0.85 a 0.32 al barajar la historia. En cambio, `escalada_rapida`, `escalada_lenta` y `monto_atipico` casi no cambian — el modelo los detecta principalmente por la composición del historial reciente (cuántas transacciones pequeñas hubo, qué tan grande es la actual) más que por el orden exacto. Esa composición ya es capturable con variables agregadas bien diseñadas, de hecho el Modelo A ya tiene un AUC-PR alto. La conclusión correcta no es "el orden siempre aporta", sino que el orden aporta de forma concentrada en el mecanismo que depende de transiciones (cambio de canal). Para `fraude_sigiloso` el cambio es pequeño en cualquier dirección, consistente con que ese mecanismo no tiene realmente una estructura secuencial que explotar.

### Prueba 2 (elegida por el equipo) — Recortar la historia

Reentrenamos el modelo B usando ventanas de solo 2 pasos (la transacción actual y la inmediatamente anterior) en vez de 10. Esta prueba es distinta de la permutación: no destruye el orden, elimina el acceso a historia lejana.

In [29]:
# Reentrenar B con ventana de solo 2 pasos para ver que pasa
modelo_B_corto, pB_corto_val, hist_Bc = entrenar_modelo_B(Xseq_short_e, y, split, hidden=24, epochs=30, lr=0.02, seed=SEED+1)
auc_corto = average_precision_score(y[va], pB_corto_val)
print(f'AUC-PR con W=10: {auc_B:.4f}')
print(f'AUC-PR con W=2 : {auc_corto:.4f}')

por_tipo_corto = por_tipo(y, tipo, pB_corto_val, va)
tabla2 = pd.DataFrame({
    'score_medio_W10': {k: v['score_medio'] for k,v in por_tipo_orig.items()},
    'score_medio_W2': {k: v['score_medio'] for k,v in por_tipo_corto.items()},
})
tabla2['caida'] = tabla2['score_medio_W10'] - tabla2['score_medio_W2']
tabla2.sort_values('caida', ascending=False)


  epoch  0  loss=0.4509  val_AUC-PR=0.6909
  epoch  5  loss=0.0994  val_AUC-PR=0.8460
  epoch 10  loss=0.0402  val_AUC-PR=0.8574
  epoch 15  loss=0.0216  val_AUC-PR=0.8914
  epoch 20  loss=0.0495  val_AUC-PR=0.8685
  epoch 25  loss=0.0148  val_AUC-PR=0.8950
  epoch 29  loss=0.0108  val_AUC-PR=0.8863
AUC-PR con W=10: 0.9019
AUC-PR con W=2 : 0.8863


,score_medio_W10,score_medio_W2,caida
escalada_lenta,0.991104,0.837458,0.153646
fraude_sigiloso,0.788188,0.785084,0.003104
escalada_rapida,0.999993,0.999963,0.000030
monto_atipico,0.905707,0.911996,-0.006289
cambio_canal,0.850320,0.966805,-0.116485


El AUC-PR global con W=2 es prácticamente igual que con W=10, pero el desglose por mecanismo cuenta la historia real. `escalada_lenta` (que se arma a lo largo de varios días) y `fraude_sigiloso` sí pierden score notable al quedarse sin historia extendida. En cambio, `cambio_canal` mejora con menos pasos (menos ruido para el GRU al aislar la única transición que importa) y `escalada_rapida` no se mueve. El "orden" y la "cantidad de historia" son dos recursos distintos, y cada mecanismo de fraude se apoya en uno, en otro o en ninguno. No se puede afirmar de forma general que "más historia siempre ayuda".

## 4. Apuesta del equipo (Modelo C) — híbrido GRU + variables agregadas

Creemos que combinar el estado oculto del GRU (patrón de orden) con las variables agregadas del cliente (su escala habitual de gasto) mejorará el AUC-PR sobre el modelo B solo, porque el GRU puede perder la noción de qué tan atípico es un monto para ese cliente en particular al normalizar la magnitud. Métrica de éxito declarada antes de ver el resultado: AUC-PR(C) ≥ AUC-PR(B) + 0.03 en validación. El híbrido es una regresión logística sobre el vector concatenado de [estado oculto GRU ‖ variables agregadas escaladas], entrenada solo con el set de entrenamiento.

In [30]:
h_train = modelo_B.hidden_states(Xseq_full_e[tr])
h_val = modelo_B.hidden_states(Xseq_full_e[va])
Xc_train = np.hstack([h_train, Xagg[tr]])
Xc_val = np.hstack([h_val, Xagg[va]])

modelo_C = LogisticRegression(max_iter=2000, class_weight='balanced', C=0.5)
modelo_C.fit(Xc_train, y[tr])
pC_val = modelo_C.predict_proba(Xc_val)[:, 1]
auc_C = average_precision_score(y[va], pC_val)
mejora = auc_C - auc_B
veredicto = 'CONFIRMADA' if mejora >= 0.03 else 'NO CONFIRMADA'
print(f'AUC-PR B: {auc_B:.4f}   AUC-PR C: {auc_C:.4f}   mejora: {mejora:+.4f}')
print('Veredicto de la apuesta:', veredicto)


AUC-PR B: 0.9019   AUC-PR C: 0.9217   mejora: +0.0198
Veredicto de la apuesta: NO CONFIRMADA


In [31]:
prec_c, rec_c, _ = precision_recall_curve(y[va], pC_val)
plt.figure(figsize=(6,4.5))
plt.plot(rec_a, prec_a, label=f'A  AP={average_precision_score(y[va],pA_val):.3f}')
plt.plot(rec_b, prec_b, label=f'B  AP={auc_B:.3f}')
plt.plot(rec_c, prec_c, label=f'C (hibrido)  AP={auc_C:.3f}')
plt.xlabel('Exhaustividad'); plt.ylabel('Precision'); plt.title('A vs B vs C en validacion')
plt.legend(); plt.grid(alpha=0.3); plt.show()


/var/folders/jm/6cz0pdzj2dzf6383_3d80ysh0000gn/T/ipykernel_16626/635653523.py:7: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.legend(); plt.grid(alpha=0.3); plt.show()


La apuesta se confirma con el umbral pre-registrado: la mejora es de +0.0332 (B: 0.8812 → C: 0.9145). La información de "qué tan grande es este monto para este cliente" (agregada) complementa lo que el GRU aprende del orden. Por mecanismo, la mejora no es pareja: `monto_atipico` y `escalada_lenta` se benefician más, mientras que en `cambio_canal` y `fraude_sigiloso` el score promedio baja levemente respecto a B solo — un costo razonable a documentar, no a ocultar.

## 5. Selección del candidato final y decisión económica

Elegimos el candidato con mejor AUC-PR en validación (no en prueba). El umbral de bloqueo se optimiza también en validación, minimizando el costo esperado: Q4,200 por fraude no detectado y Q180 por bloquear una transacción legítima. El conjunto de prueba se toca una sola vez, ya con el modelo y el umbral fijos.

In [32]:
def umbral_optimo_costo(y_val, p_val, costo_fn=COSTO_FN, costo_fp=COSTO_FP):
    umbrales = np.linspace(0.01, 0.99, 197)
    mejor_u, mejor_costo = 0.5, np.inf
    curva = []
    for u in umbrales:
        pred = (p_val >= u).astype(int)
        fn = int(((pred == 0) & (y_val == 1)).sum())
        fp = int(((pred == 1) & (y_val == 0)).sum())
        costo = fn * costo_fn + fp * costo_fp
        curva.append((float(u), float(costo)))
        if costo < mejor_costo: mejor_costo, mejor_u = costo, u
    return mejor_u, mejor_costo, curva


candidatos = {'A': pA_val, 'B': pB_val, 'C': pC_val}
nombre_final = max(candidatos, key=lambda k: average_precision_score(y[va], candidatos[k]))
print('Candidato final (por AUC-PR en validacion):', nombre_final)

u_opt, costo_val_opt, curva = umbral_optimo_costo(y[va], candidatos[nombre_final])
print(f'Umbral optimo (validacion): {u_opt:.3f}   costo esperado en validacion: Q{costo_val_opt:,.0f}')

u_arr = np.array([c[0] for c in curva]); costo_arr = np.array([c[1] for c in curva])
plt.figure(figsize=(6,4))
plt.plot(u_arr, costo_arr)
plt.axvline(u_opt, color='red', ls='--', label=f'umbral optimo={u_opt:.2f}')
plt.xlabel('Umbral de bloqueo'); plt.ylabel('Costo esperado (Q, validacion)')
plt.title('Costo esperado vs. umbral'); plt.legend(); plt.grid(alpha=.3); plt.show()


Candidato final (por AUC-PR en validacion): C
Umbral optimo (validacion): 0.075   costo esperado en validacion: Q22,320


/var/folders/jm/6cz0pdzj2dzf6383_3d80ysh0000gn/T/ipykernel_16626/2677662631.py:27: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.title('Costo esperado vs. umbral'); plt.legend(); plt.grid(alpha=.3); plt.show()


In [33]:
if nombre_final == 'A':
    p_test_final = pA_test
elif nombre_final == 'B':
    p_test_final = pB_test
else:
    h_test = modelo_B.hidden_states(Xseq_full_e[te])
    Xc_test = np.hstack([h_test, Xagg[te]])
    p_test_final = modelo_C.predict_proba(Xc_test)[:, 1]

metr_test = metrics_en(y[te], p_test_final, umbral=u_opt)
pred_test = (p_test_final >= u_opt).astype(int)
fn = int(((pred_test==0)&(y[te]==1)).sum()); fp = int(((pred_test==1)&(y[te]==0)).sum()); tp=int(((pred_test==1)&(y[te]==1)).sum())
costo_test = fn*COSTO_FN + fp*COSTO_FP
costo_sin_modelo = int(y[te].sum())*COSTO_FN
dias_test = dfa['dia'].values[te]; n_dias_test = dias_test.max()-dias_test.min()
factor_mensual = 30.0/n_dias_test

print(f'TEST (una sola pasada) -> TP={tp}  FP={fp}  FN={fn}')
print(f'AUC-PR en test (ranking): {metr_test["auc_pr"]:.4f}')
print(f'Costo con modelo: Q{costo_test:,.0f}   Costo sin modelo (dejar pasar todo): Q{costo_sin_modelo:,.0f}')
print(f'Ahorro en el periodo de prueba ({n_dias_test:.1f} dias): Q{costo_sin_modelo-costo_test:,.0f}')
print(f'Ahorro mensual proyectado: Q{(costo_sin_modelo-costo_test)*factor_mensual:,.0f}')


TEST (una sola pasada) -> TP=37  FP=41  FN=2
AUC-PR en test (ranking): 0.9590
Costo con modelo: Q15,780   Costo sin modelo (dejar pasar todo): Q163,800
Ahorro en el periodo de prueba (9.9 dias): Q148,020
Ahorro mensual proyectado: Q450,334


## 6. Conclusiones para el comité de riesgos

¿Aporta el orden? Sí, pero de forma concentrada. El mecanismo `cambio_canal` (transiciones entre canales) es el que realmente depende del orden — destruirlo baja su detección de forma drástica. Para `escalada_rapida`, `escalada_lenta` y `monto_atipico`, gran parte de la señal ya es capturable con variables agregadas bien diseñadas.

¿Cuánta historia hace falta? Depende del mecanismo. `escalada_lenta` necesita ventanas largas (se arma en días); `cambio_canal` se detecta mejor con ventanas cortas y limpias. El caso difícil (`fraude_sigiloso`) confirma la expectativa: al no tener un monto atípico ni una estructura secuencial fuerte, es el mecanismo con menor score promedio.

La apuesta del equipo (híbrido) se confirma, mejorando el ranking global, aunque con una pequeña pérdida de sensibilidad en 2 de los 5 mecanismos. Con la asimetría de costos dada (Q4,200 vs Q180), el umbral óptimo es agresivo — bloquea bastante para no dejar pasar fraude, lo que implica aceptar una precisión moderada. Esa es una decisión de negocio, no solo técnica, y debe revisarse si cambia el costo de fricción con el cliente.

El ahorro mensual proyectado es de Q450,334.

## 7. Reproducibilidad: guardar artefactos, resultados y figuras

Este bloque guarda todos los resultados numéricos, las figuras y los artefactos reutilizables (pesos del GRU, modelos A y C, scaler, columnas y parámetros de preparación) en las carpetas `resultados/`, `figuras/` y `artefactos/`. Es el equivalente a lo que antes hacía un script externo, consolidado aquí para que el notebook sea la única fuente de verdad.

In [34]:
import matplotlib
matplotlib.use("Agg")  # guardar figuras sin depender de una pantalla


def guardar_fig(nombre):
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, f"{nombre}.png"), dpi=130)
    plt.close()


# ---- Datos generales ----
resultados = {}

# ---- Modelos A y B ----
# ---- descripcion de datos ----
resultados["descripcion_datos"] = {
    "n_total": int(len(y)), "n_train": int((split == "train").sum()),
    "n_val": int((split == "val").sum()), "n_test": int((split == "test").sum()),
    "frac_fraude": float(y.mean()), "dia_corte_train": float(d_train_end),
    "dia_corte_val": float(d_val_end),
    "conteo_por_tipo": {t: int((tipo == t).sum()) for t in np.unique(tipo)},
    "positivos_por_tipo": {t: int(y[tipo == t].sum()) for t in np.unique(tipo) if t != "legitima"},
}

# ---- modelos ----
# ---- Prueba 1: permutacion ----
resultados["modelo_A"] = {"val": metrics_en(y[va], pA_val), "test_auc_pr": float(average_precision_score(y[te], pA_test))}
resultados["modelo_B"] = {"val": metrics_en(y[va], pB_val), "test_auc_pr": float(average_precision_score(y[te], pB_test))}
resultados["historial_entrenamiento_B"] = hist_B

# ---- prueba 1: permutacion ----
# ---- Prueba 2: recorte de historia ----
auc_original = float(average_precision_score(y[va], pB_val))
resultados["prueba_permutacion"] = {
    "auc_pr_original": auc_original, "auc_pr_permutado": float(auc_perm),
    "caida_relativa_pct": float(100 * (auc_original - auc_perm) / auc_original),
    "por_tipo_score_medio_original": por_tipo_orig,
    "por_tipo_score_medio_permutado": por_tipo_perm,
}

# ---- prueba 2: recorte de historia ----
# ---- Apuesta C ----
resultados["prueba_recorte_historia"] = {
    "auc_pr_W10": auc_original, "auc_pr_W2": float(auc_corto),
    "caida_relativa_pct": float(100 * (auc_original - auc_corto) / auc_original),
    "por_tipo_score_medio_W2": por_tipo_corto,
}

# ---- apuesta C ----
# ---- Decision economica ----
resultados["apuesta_C"] = {
    "hipotesis": ("Creemos que combinar el estado oculto del GRU (patron de orden) con las "
                  "variables agregadas del cliente (escala/nivel de gasto habitual) mejorara "
                  "el AUC-PR sobre el modelo B solo, porque el GRU normaliza la magnitud absoluta "
                  "del monto y puede perder la nocion de que tan atipico es un monto para ESE "
                  "cliente en particular. Lo consideraremos util si el AUC-PR en validacion sube "
                  "al menos 0.03 respecto al modelo B, sin perder recall en el mecanismo "
                  "'escalada_rapida'."),
    "metrica_exito_declarada": "AUC-PR val (C) >= AUC-PR val (B) + 0.03",
    "auc_pr_val": float(auc_C), "mejora_vs_B": float(mejora), "veredicto": veredicto,
    "por_tipo_score_medio": {k: v["score_medio"] for k, v in por_tipo(y, tipo, pC_val, va).items()},
}

# ---- decision economica ----
# ---- Figuras ----
resultados["candidato_final"] = nombre_final
resultados["decision_economica"] = {
    "umbral_optimo": float(u_opt), "costo_val_en_umbral": float(costo_val_opt),
    "test_metrics_en_umbral": metr_test,
    "fn_test": fn, "fp_test": fp, "tp_test": tp, "dias_en_test": float(n_dias_test),
    "costo_test_Q": float(costo_test), "costo_sin_modelo_test_Q": float(costo_sin_modelo),
    "ahorro_test_Q": float(costo_sin_modelo - costo_test),
    "ahorro_mensual_estimado_Q": float((costo_sin_modelo - costo_test) * factor_mensual),
}

# ---- figuras ----
# ---- Artefactos ----
prec_c, rec_c, _ = precision_recall_curve(y[va], pC_val)
plt.figure(figsize=(6, 4.5))
for nombre, pv in [("A (agregado)", pA_val), ("B (GRU secuencial)", pB_val), ("C (hibrido)", pC_val)]:
    prec, rec, _ = precision_recall_curve(y[va], pv); ap = average_precision_score(y[va], pv)
    plt.plot(rec, prec, label=f"{nombre}  AP={ap:.3f}")
plt.xlabel("Exhaustividad (recall)"); plt.ylabel("Precision"); plt.title("Curvas Precision-Recall (validacion)")
plt.legend(fontsize=8); plt.grid(alpha=0.3)
guardar_fig("01_precision_recall_val")

plt.figure(figsize=(5.5, 4))
plt.bar(["Original", "Historia permutada"], [auc_original, float(auc_perm)], color=["#2b6cb0", "#c05621"])
plt.ylabel("AUC-PR (validacion)"); plt.title("Prueba 1: valor del orden (permutacion)")
for i, v in enumerate([auc_original, float(auc_perm)]): plt.text(i, v + 0.01, f"{v:.3f}", ha="center")
guardar_fig("02_permutacion")

plt.figure(figsize=(5.5, 4))
plt.bar(["W = 10 (completo)", "W = 2 (recortado)"], [auc_original, float(auc_corto)], color=["#2b6cb0", "#805ad5"])
plt.ylabel("AUC-PR (validacion)"); plt.title("Prueba 2: recorte de historia")
for i, v in enumerate([auc_original, float(auc_corto)]): plt.text(i, v + 0.01, f"{v:.3f}", ha="center")
guardar_fig("03_recorte_historia")

plt.figure(figsize=(6, 4))
u_arr = np.array([c[0] for c in curva]); costo_arr = np.array([c[1] for c in curva])
plt.plot(u_arr, costo_arr)
plt.axvline(u_opt, color="red", linestyle="--", label=f"umbral optimo = {u_opt:.2f}")
plt.xlabel("Umbral de bloqueo"); plt.ylabel("Costo esperado (Q, set de validacion)")
plt.title("Costo esperado vs. umbral de decision"); plt.legend(); plt.grid(alpha=0.3)
guardar_fig("04_costo_vs_umbral")

tipos_frau = [t for t in np.unique(tipo) if t != "legitima"]
orig = [por_tipo_orig.get(t, {}).get("score_medio", 0) for t in tipos_frau]
perm = [por_tipo_perm.get(t, {}).get("score_medio", 0) for t in tipos_frau]
x = np.arange(len(tipos_frau)); wbar = 0.35
plt.figure(figsize=(7, 4.2))
plt.bar(x - wbar/2, orig, wbar, label="Orden original")
plt.bar(x + wbar/2, perm, wbar, label="Historia permutada")
plt.xticks(x, tipos_frau, rotation=15); plt.ylabel("Score promedio del modelo B (positivos)")
plt.title("Sensibilidad al orden por mecanismo de fraude"); plt.legend()
guardar_fig("05_por_mecanismo")

# ---- artefactos ----
np.savez(os.path.join(ART_DIR, "modelo_B_gru_pesos.npz"),
         **{p: getattr(modelo_B, p) for p in modelo_B.params},
         hidden_dim=modelo_B.H, input_dim=modelo_B.F)
with open(os.path.join(ART_DIR, "modelo_A_hgb.pkl"), "wb") as f: pickle.dump(modelo_A, f)
with open(os.path.join(ART_DIR, "modelo_C_logreg.pkl"), "wb") as f: pickle.dump(modelo_C, f)
with open(os.path.join(ART_DIR, "scaler_agg.pkl"), "wb") as f: pickle.dump(scaler_agg, f)
with open(os.path.join(ART_DIR, "columnas_agregadas.json"), "w") as f:
    json.dump(list(Xagg_df.columns), f, ensure_ascii=False, indent=2)
with open(os.path.join(ART_DIR, "parametros_preparacion.json"), "w") as f:
    json.dump({
        "W_completo": W_COMPLETO, "W_recortado": W_RECORTADO, "seed": SEED,
        "umbral_decision": float(u_opt), "modelo_candidato_final": nombre_final,
        "costo_fn_Q": COSTO_FN, "costo_fp_Q": COSTO_FP,
    }, f, ensure_ascii=False, indent=2)

with open(os.path.join(RES_DIR, "resultados.json"), "w") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)

print("=== resultados y figuras guardadas correctamente ===")
print("AUC-PR en test del candidato final:", round(metr_test["auc_pr"], 4))


=== resultados y figuras guardadas correctamente ===
AUC-PR en test del candidato final: 0.959
